# 05 · Kümeleme (Gözetimsiz Öğrenme)
Etiket kullanmadan benzer gözlemleri gruplamak. **Iris** veri setinde (150 çiçek, 4 ölçüm) türleri bilmiyormuş gibi davranıp grupları bulmaya çalışacağız, sonra gerçek türlerle karşılaştıracağız.

| Algoritma | Fikir | Küme sayısı gerekir mi? |
|---|---|---|
| **K-Means** | Noktaları en yakın merkeze ata, merkezleri güncelle | Evet |
| **DBSCAN** | Yoğun bölgeleri küme, seyrek noktaları gürültü say | Hayır |
| **Hiyerarşik** | En yakın kümeleri adım adım birleştir | Dendrogramdan seçilir |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage

## 1. Veri ve ölçekleme

In [ ]:
iris = load_iris(as_frame=True)
df = iris.data
gercek_tur = iris.target
print("Eksik değer:", df.isna().sum().sum())
display(df.describe().round(2))

X = StandardScaler().fit_transform(df)                # gözetimsiz → test seti yok
X_2d = PCA(n_components=2).fit_transform(X)            # görselleştirme için 2 boyut

**Yorum:** Değişkenlerin ölçekleri farklı (petal width 0.1–2.5, sepal length 4.3–7.9). Mesafe tabanlı algoritmalarda büyük ölçekli değişken baskın olmasın diye standartlaştırdık.

In [ ]:
sns.pairplot(df, diag_kind="kde", plot_kws={"alpha": 0.5, "color": "tab:gray"}, height=2.2); plt.show()

**Yorum:** Etiket rengi olmadan bile petal ölçümlerinde belirgin bir ayrı grup (setosa) görülüyor; diğer iki tür iç içe geçmiş.

## 2. K-Means – k seçimi (Dirsek + Silhouette)

In [ ]:
K = range(2, 9)
inertia, sil = [], []
for k in K:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(K, inertia, "o-", color="tab:blue"); ax[0].set(title="Dirsek Yöntemi", xlabel="k", ylabel="Inertia")
ax[1].plot(K, sil, "o-", color="tab:orange");   ax[1].set(title="Silhouette (yüksek = iyi)", xlabel="k")
plt.tight_layout(); plt.show()
pd.DataFrame({"k": list(K), "silhouette": sil}).round(3).T

**Yorum:** Silhouette skoru genellikle k=2'de en yüksektir (setosa vs diğerleri çok net ayrılır), dirsek ise k=3 civarındadır. Bu, **metriklerin her zaman 'gerçek' grup sayısını vermediğini** gösterir; alan bilgisi ile birlikte karar verilmelidir. Biz 3 küme ile devam edelim.

In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(*X_2d.T, c=km.labels_, cmap="viridis", s=30); ax[0].set_title("K-Means Kümeleri (PCA 2D)")
ax[1].scatter(*X_2d.T, c=gercek_tur, cmap="Set1", s=30);   ax[1].set_title("Gerçek Türler (PCA 2D)")
plt.show()
print("Adjusted Rand Index (gerçekle uyum, 1=mükemmel):", round(adjusted_rand_score(gercek_tur, km.labels_), 3))

In [ ]:
df.assign(kume=km.labels_).groupby("kume").mean().round(2)

**Yorum:** Kümelerin ortalama özellikleri onları tanımlamamızı sağlar: küçük petalli küme setosa'ya, büyük petalli küme virginica'ya karşılık gelir. Versicolor ve virginica sınırında bazı karışmalar olur.

## 3. DBSCAN

In [ ]:
# eps seçimi için k-mesafe grafiği
mesafe, _ = NearestNeighbors(n_neighbors=5).fit(X).kneighbors(X)
plt.figure(figsize=(7, 3.5)); plt.plot(np.sort(mesafe[:, -1]), color="tab:red")
plt.title("5. komşuya uzaklık (dirsek → eps)"); plt.ylabel("mesafe"); plt.show()

In [ ]:
db = DBSCAN(eps=0.8, min_samples=5).fit(X)
etiket = db.labels_
print("Küme sayısı:", len(set(etiket)) - (1 if -1 in etiket else 0), "| Gürültü noktası:", (etiket == -1).sum())
plt.figure(figsize=(6.5, 5)); plt.scatter(*X_2d.T, c=etiket, cmap="tab10", s=30)
plt.title("DBSCAN (-1 = gürültü)"); plt.show()

**Yorum:** DBSCAN küme sayısını kendisi belirler ve aykırı noktaları `-1` (gürültü) olarak işaretler. İç içe geçmiş versicolor/virginica'yı tek küme olarak görmesi doğaldır — yoğunluk açısından gerçekten bitişikler.

## 4. Hiyerarşik Kümeleme

In [ ]:
plt.figure(figsize=(14, 5))
dendrogram(linkage(X, method="ward"), truncate_mode="lastp", p=30, color_threshold=7)
plt.title("Dendrogram (Ward)"); plt.ylabel("mesafe"); plt.show()

agg = AgglomerativeClustering(n_clusters=3, linkage="ward").fit(X)
print("Hiyerarşik ARI:", round(adjusted_rand_score(gercek_tur, agg.labels_), 3))

**Yorum:** Dendrogramda en uzun dikey çizgiyi kesen yatay çizgi doğal küme sayısını önerir. Ward bağlantısı, K-Means'e benzer kompakt kümeler üretir.

## Özet
### Data Analysis Key Findings
- K-Means ve hiyerarşik kümeleme, etiket görmeden 3 iris türünü büyük ölçüde (ARI ≈ 0.6) geri buldu; setosa kusursuz ayrılırken versicolor/virginica kısmen karıştı.
- Silhouette skoru k=2'yi önerdi — metrikler alan bilgisinin yerini tutmaz.
- DBSCAN küme sayısı istemeden yoğunluk temelli gruplar ve gürültü noktaları buldu.
### Insights or Next Steps
- Müşteri segmentasyonu gibi gerçek problemlerde kategorik değişkenler için One-Hot/Ordinal encoding ekleyin.